# The cost of choosing a model

## Statistical evidence, validation stability and uncertain intervention costs

The earlier notebooks showed that:
- PCA variance, regression inference and predictive relevance are distinct sources of information.
- Missing predictors and contaminated outcomes can affect prediction and economic decisions differently.

This study asks a further question: **even when candidate models are fixed, how much does the choice of validation split and decision cost influence which model we select?**

We distinguish three training-only selection objectives:

1. Lowest validation RMSE;
2. Highest validation net benefit at a nominal cost;
3. Highest validation net benefit averaged over a plausible cost grid.

Each objective is evaluated once on a single split and again across repeated validation splits. The selected model is refitted on all training observations and assessed on an independent test set. We do not choose a model after seeing the test results.

## 1. The statistical and decision-theoretic structure

We interpret the simulated response \(Y\) as a *potential net benefit before paying the cost* of an action. Under a known cost \(c\), a prediction \(\widehat m(x)\) induces the decision

\[
a_c(x)=\mathbb{1}\{\widehat m(x)>c\}.
\]

For the full-information conditional mean \(m(x)=E[Y\mid X=x]\), the simulated oracle acts when \(m(x)>c\). Expected regret is

\[
R_c(\widehat m)=E[(m(X)-c)_+]-
E[(m(X)-c)\mathbb{1}\{\widehat m(X_{\rm obs})>c\}].
\]

At model-selection time, the future cost is uncertain. We place a discrete uniform distribution on four plausible costs \(C\in\{0.2,0.6,1.0,1.4\}\) and compare selection based on

\[
\widehat U_{\rm grid}(\widehat m)
=\frac{1}{4}\sum_{c\in\mathcal C}
\frac{1}{n_{\rm val}}\sum_{i\in {\rm val}}
(Y_i-c)\mathbb{1}\{\widehat m(X_i)>c\}.
\]

The cost is revealed **before each decision**, so the threshold can adapt to the realised value. If the action must be committed before the cost becomes known, this is a different optimisation problem.

All utilities are calculated using *observed training-validation outcomes* during selection. We use known synthetic conditional means only to evaluate final test regret. Nothing here identifies causal treatment effects.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cost_selection import (
    DEFAULT_COSTS,
    paired_policy_regret,
    run_cost_selection_experiment,
)

REPEATS: int = int(os.environ.get("SELECTION_REPEATS", "6"))
SPLITS: int = int(os.environ.get("SELECTION_SPLITS", "3"))
TRAIN_SIZES: tuple[int, ...] = tuple(
    int(item.strip())
    for item in os.environ.get("SELECTION_SIZES", "180").split(",")
)
SMOKE: bool = os.environ.get("SELECTION_SMOKE", "0") == "1"
SCENARIOS = ("low_variance_signal",) if SMOKE else (
    "low_variance_signal", "high_variance_signal"
)
STRESSES = ("clean", "mar_outliers")  # Paired stress comparison.
NOMINAL_COST: float = 0.6

if REPEATS < 1 or SPLITS < 2 or not TRAIN_SIZES or min(TRAIN_SIZES) < 64:
    raise ValueError("Invalid selection settings")
print(
    f"Repeats: {REPEATS}; internal splits: {SPLITS}; "
    f"training sizes: {TRAIN_SIZES}; cost grid: {DEFAULT_COSTS}"
)

## 2. Experimental design

We reuse the original correlated-predictor simulation with two signal mechanisms: a low-variance contrast and a high-variance sum. For each independent replicate, we evaluate both clean data and a MAR-plus-outliers stress condition built from the same underlying observations.

Each candidate estimator uses its own training-only imputation, scaling and internal hyperparameter tuning. We compare six candidate estimators: OLS, ridge, elastic net, Huber, 95%-variance PCR and two-component PLS. Under the stressed condition, the observed validation outcomes themselves can be contaminated; neither utility selection nor RMSE selection is guaranteed to be robust.

Repeated validation splits overlap in training data and must not be treated as independent statistical replications. Final reported uncertainty is based on *independent regenerated datasets*, not on those internal splits.

In [ ]:
evaluation, audit = run_cost_selection_experiment(
    repeats=REPEATS,
    n_train_values=TRAIN_SIZES,
    n_test=500,
    costs=DEFAULT_COSTS,
    nominal_cost=NOMINAL_COST,
    split_count=SPLITS,
    seed=2026,
    scenarios=SCENARIOS,
    stresses=STRESSES,
)
display(audit.head(12))
display(evaluation.head(12))
assert (evaluation["regret"] >= -1e-9).all()
assert np.isfinite(evaluation[["test_rmse", "regret", "expected_profit"]]).all().all()

## 3. How often does the selected model depend on the objective?

The single-split policy uses just the first internal split. The repeated-split policy averages validation evidence from multiple splits. Both share the same independent test set.

For repeated splits, agreement is the fraction of individual split winners matching the aggregate winner. This is *descriptive sensitivity*, not an unbiased selection-probability estimator: repeated holdouts overlap and are dependent.

Review candidate selection before comparing average economic regret; low regret should not retroactively be used to change model-selection criteria.

In [ ]:
selection_frequency = (
    evaluation.drop_duplicates(
        ["scenario", "stress", "n_train", "replicate", "policy"]
    )
    .groupby(["stress", "policy", "selected_model"], as_index=False)
    .size()
    .rename(columns={"size": "times_selected"})
    .sort_values(["stress", "policy", "times_selected"],
                 ascending=[True, True, False])
)
display(selection_frequency)
agreement = (
    evaluation.loc[evaluation["protocol"] == "Repeated splits"]
    .drop_duplicates(["scenario", "stress", "n_train", "replicate", "policy"])
    .groupby(["stress", "criterion"], as_index=False)
    .agg(
        mean_split_agreement=("split_agreement", "mean"),
        min_split_agreement=("split_agreement", "min"),
    )
)
display(agreement.round(3))

## 4. Decision cost as a sensitivity parameter

Model choice based on RMSE, a single nominal cost and a distribution of possible future costs may disagree. We report regret at *each* cost instead of collapsing it prematurely.

Because the grid is averaged uniformly during training selection, the study assumes each listed cost is equally plausible. This can later be generalised to externally justified weights or continuous distributions.

In [ ]:
summary = (
    evaluation.groupby(
        ["scenario", "stress", "n_train", "cost", "policy"],
        as_index=False
    )
    .agg(mean_regret=("regret", "mean"),
         mean_rmse=("test_rmse", "mean"),
         mean_expected_profit=("expected_profit", "mean"))
)
focus_size = min(TRAIN_SIZES, key=lambda n: abs(n - 180))
focus = summary.loc[
    (summary["scenario"] == "low_variance_signal")
    & (summary["n_train"] == focus_size)
]
display(focus.sort_values(["stress", "cost", "mean_regret"]).round(3))

In [ ]:
highlight = [
    "Single split / Nominal utility",
    "Repeated splits / Nominal utility",
    "Repeated splits / Cost-grid utility",
    "Repeated splits / RMSE",
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, stress in zip(axes, STRESSES):
    frame = focus[focus["stress"] == stress]
    for policy in highlight:
        series = frame[frame["policy"] == policy].sort_values("cost")
        ax.plot(
            series["cost"], series["mean_regret"],
            marker="o", label=policy
        )
    ax.set(
        xlabel="Cost revealed at decision time",
        title=stress.replace("_", " ").capitalize(),
    )
axes[0].set_ylabel("Mean expected regret per test observation")
axes[1].legend(loc="best", fontsize="small")
fig.suptitle("Decision regret versus realised cost")
fig.tight_layout()
plt.show()

## 5. Paired policy comparisons

We pair the independent-test regret of alternative selection policies on the *same simulated dataset and same realised cost*. A negative change means the comparison policy has less regret.

Bootstrap intervals resample complete independent simulation replicates. They do not treat validation splits, test observations or cost scenarios as independent experimental units. With only a few simulation replicates these intervals are exploratory, not evidence of reliable superiority.

In [ ]:
if REPEATS >= 3:
    baseline_policy = "Single split / Nominal utility"
    candidates = (
        "Repeated splits / Nominal utility",
        "Repeated splits / Cost-grid utility",
        "Repeated splits / RMSE",
    )
    reports: list[pd.DataFrame] = []
    for candidate in candidates:
        report = paired_policy_regret(
            evaluation,
            reference_policy=baseline_policy,
            comparison_policy=candidate,
            bootstrap_draws=600, seed=77,
        )
        reports.append(report)
    paired = pd.concat(reports, ignore_index=True)
    display(
        paired.loc[
            (paired["scenario"] == "low_variance_signal")
            & (paired["n_train"] == focus_size)
        ].sort_values(["stress", "cost", "mean_regret_change"])
        .round(3)
    )
else:
    print("Smoke verification: independent replicates too few for a useful interval.")
    paired = None

## 6. How much can we conclude?

This is a controlled comparison, not a universal algorithm ranking. The analysis separates four information sources:

- Feature geometry and coefficient uncertainty, explored in the original notebook.
- Predictive behaviour under missingness and contamination, explored in the robustness notebook.
- Stability of candidate selection when training observations are re-partitioned.
- Sensitivity of expected decision regret to the action's cost.

A repeated validation procedure can reduce dependence on one split, but it does not guarantee a more useful choice. Selection using a cost distribution aligns the scoring objective with expected economic payoff *if* the distribution is relevant, the benefit is interpretable and there is no hidden confounding.

Two limitations deserve emphasis. First, test regret under MAR is measured against a full-information oracle, so even an ideal observed-data model has unavoidable regret. Second, cost-weighted selection optimises a *predictive* benefit surrogate; a real policy would need a causal outcome definition.

### Research extensions

1. Nested, repeated cross-validation with unbiased external comparisons and larger independent Monte Carlo samples.
2. Stressing the assumed cost distribution and allowing heterogeneous individual costs.
3. Model-uncertainty and selection-uncertainty decomposition, calibration and explicit uncertainty quantification.
4. MNAR sensitivity and alternative imputation models.
5. Real economic decisions with an explicit utility function and an identifiable causal design.

Read the project README for the experiment contract and full references.